# Phase 4: Modellierung

**Ziel:** Baseline-Modell trainieren, verbessern, Hyperparameter tunen.

**StackFuel-Themen:** Mathematik · ML-Grundlagen

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

# Daten laden (aus Notebook 02)
X_train = np.load('../data/processed/X_train.npy')
X_val   = np.load('../data/processed/X_val.npy')
y_train = np.load('../data/processed/y_train.npy')
y_val   = np.load('../data/processed/y_val.npy')
classes = np.load('../data/processed/classes.npy')

print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'Klassen: {classes}')

## 4.1 Baseline: Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=200, random_state=42)
lr.fit(X_train, y_train)

acc_train = lr.score(X_train, y_train)
acc_val   = lr.score(X_val, y_val)

print(f'Logistic Regression')
print(f'  Train Accuracy: {acc_train:.3f}')
print(f'  Val   Accuracy: {acc_val:.3f}')

## 4.2 Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

acc_train_rf = rf.score(X_train, y_train)
acc_val_rf   = rf.score(X_val, y_val)

print(f'Random Forest')
print(f'  Train Accuracy: {acc_train_rf:.3f}')
print(f'  Val   Accuracy: {acc_val_rf:.3f}')

## 4.3 Modelle vergleichen

In [ ]:
modelle = {
    'Logistic Regression': lr,
    'Random Forest': rf,
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5)
}

ergebnisse = []
for name, model in modelle.items():
    if name == 'KNN (k=5)':
        model.fit(X_train, y_train)  # KNN wurde noch nicht gefittet
    scores = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy')
    ergebnisse.append({'Modell': name, 'CV Accuracy Ø': scores.mean().round(3), 'Std': scores.std().round(3)})

import pandas as pd
df_results = pd.DataFrame(ergebnisse).sort_values('CV Accuracy Ø', ascending=False)
print(df_results.to_string(index=False))

## 4.4 Hyperparameter-Tuning mit GridSearchCV

In [ ]:
# Kleines Grid für Random Forest
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid,
    cv=3,
    scoring='accuracy',
    verbose=1
)
grid_search.fit(X_train, y_train)

print(f'Beste Parameter: {grid_search.best_params_}')
print(f'Beste CV Accuracy: {grid_search.best_score_:.3f}')

best_model = grid_search.best_estimator_

**Beobachtung:**  
_Welche Parameter haben am besten funktioniert? Gibt es Overfitting?_